# ESMC Fine-Tuning - Colab Version

Fine-tune ESMC for custom classification tasks using parameter-efficient LoRA adapters.

**What it does:**
- Train ESMC with frozen backbone + LoRA adapters (efficient)
- Multi-class classification (EC enzyme classification as example)
- Custom training loop with validation every 250 steps
- Save fine-tuned adapters for reuse
- Track metrics: loss, accuracy, confusion matrix

**Key Features:**
- PEFT LoRA: Only ~0.4% of parameters trainable (memory efficient)
- Batch size 8 training, 16 eval (fits on Colab A100)
- bfloat16 mixed precision (faster training)
- ~4 minutes for 1000 training steps


In [ ]:
!pip install -q torch transformers peft accelerate einops pandas numpy tqdm matplotlib scikit-learn

In [ ]:
!pip install -q git+https://github.com/Biohub/esm.git@main

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.optim import AdamW
from torch.cuda.amp import autocast
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import json
from tqdm import tqdm
from IPython.display import display, Markdown

from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
# Training configuration
CONFIG = {
    # Model
    "model_name": "biohub/ESMC-300M",
    "num_labels": 7,  # EC classification: 7 enzyme classes
    
    # LoRA
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.01,
    "lora_target_modules": ["out_proj"],  # Attention output layers
    
    # Training
    "learning_rate": 1e-4,
    "num_train_steps": 1000,  # ~4 min on A100; increase for full training
    "train_batch_size": 8,
    "eval_batch_size": 16,
    "max_seq_length": 1024,
    "eval_every_n_steps": 250,
    "val_fraction": 0.01,  # 1% for validation
}

# Device & paths
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "/content/esmc_finetuned"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# EC class names (for enzyme classification example)
EC_CLASSES = [
    "Oxidoreductases",
    "Transferases",
    "Hydrolases",
    "Lyases",
    "Isomerases",
    "Ligases",
    "Translocases"
]

print(f"Device: {DEVICE}")
print(f"Model: {CONFIG['model_name']}")
print(f"LoRA config: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, dropout={CONFIG['lora_dropout']}")
print(f"Training: {CONFIG['num_train_steps']} steps, batch_size={CONFIG['train_batch_size']}")

## Data Preparation

In [ ]:
# Create sample training dataset (EC classification)
# In practice, load from CARE dataset or your own data

def create_sample_dataset(num_samples: int = 500) -> pd.DataFrame:
    """Create synthetic enzyme dataset for demonstration."""
    np.random.seed(42)
    
    # Sample sequences (use real proteins for actual training)
    sample_sequences = [
        "MKVLIVAALLLAVGLAFWECEKRKYQCPEKPQE",
        "MDVFMGVGVVDAKALVDYLVPGQDTAV",
        "MVSGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTG",
        "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
    ]
    
    sequences = []
    labels = []
    
    for i in range(num_samples):
        # Random sequence (or pick from samples and extend)
        base_seq = np.random.choice(sample_sequences)
        # Mutate slightly for variety
        seq = list(base_seq)
        for _ in range(np.random.randint(0, 3)):
            idx = np.random.randint(0, len(seq))
            seq[idx] = np.random.choice(list("ACDEFGHIKLMNPQRSTVWY"))
        sequences.append("".join(seq))
        
        # Random EC label (0-6)
        labels.append(np.random.randint(0, 7))
    
    return pd.DataFrame({"sequence": sequences, "label": labels})

# Or load your own dataset:
# df_train = pd.read_csv("your_training_data.csv")  # columns: sequence, label

df_data = create_sample_dataset(num_samples=500)
print(f"Created sample dataset: {len(df_data)} sequences")
print(f"Label distribution:\n{df_data['label'].value_counts().sort_index()}")
print(f"Sequence length stats: {df_data['sequence'].str.len().describe()}")

In [ ]:
# Stratified train/val split
from sklearn.model_selection import train_test_split

val_size = max(1, int(len(df_data) * CONFIG["val_fraction"]))

df_train, df_val = train_test_split(
    df_data,
    test_size=val_size,
    stratify=df_data["label"],
    random_state=42
)

print(f"Train set: {len(df_train)} samples")
print(f"Val set: {len(df_val)} samples")
print(f"\nTraining label distribution:\n{df_train['label'].value_counts().sort_index()}")

## Load Model & Setup LoRA

In [ ]:
print(f"Loading {CONFIG['model_name']}...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
print(f"✓ Tokenizer loaded. Vocab size: {len(tokenizer)}")

# Load base model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=CONFIG["num_labels"],
    device_map="auto"
)
print(f"✓ Model loaded. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["lora_target_modules"],
    task_type="SEQ_CLS",
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")
print(model.print_trainable_parameters())

## Training Loop

In [ ]:
def make_batch(sequences: List[str], labels: List[int], tokenizer, max_len: int, device):
    """Tokenize and prepare batch."""
    encodings = tokenizer(
        sequences,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    
    batch = {
        "input_ids": encodings["input_ids"].to(device),
        "attention_mask": encodings["attention_mask"].to(device),
        "labels": torch.tensor(labels, device=device)
    }
    return batch

def evaluate(model, df_eval, tokenizer, config, device):
    """Evaluate on validation set."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_start in range(0, len(df_eval), config["eval_batch_size"]):
            batch_end = min(batch_start + config["eval_batch_size"], len(df_eval))
            batch_df = df_eval.iloc[batch_start:batch_end]
            
            batch = make_batch(
                batch_df["sequence"].tolist(),
                batch_df["label"].tolist(),
                tokenizer,
                config["max_seq_length"],
                device
            )
            
            with autocast(dtype=torch.bfloat16):
                outputs = model(**batch)
            
            loss = outputs.loss
            logits = outputs.logits
            
            total_loss += loss.item() * (batch_end - batch_start)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch_end - batch_start
    
    model.train()
    return total_loss / total, correct / total

print("✓ Utility functions defined")

In [ ]:
# Setup optimizer
optimizer = AdamW(model.parameters(), lr=CONFIG["learning_rate"])
model.train()

# Training tracking
train_history = {"step": [], "loss": [], "accuracy": []}
val_history = {"step": [], "loss": [], "accuracy": []}

print(f"\n🚀 Starting training for {CONFIG['num_train_steps']} steps...\n")

# Create epoch sampler
num_epochs = max(1, (CONFIG["num_train_steps"] + len(df_train) - 1) // len(df_train))
steps_per_epoch = len(df_train) // CONFIG["train_batch_size"]

step = 0
for epoch in range(num_epochs):
    if step >= CONFIG["num_train_steps"]:
        break
    
    # Shuffle and iterate over training data
    indices = np.random.permutation(len(df_train))
    
    pbar = tqdm(range(0, len(indices), CONFIG["train_batch_size"]), desc=f"Epoch {epoch+1}")
    for batch_start in pbar:
        if step >= CONFIG["num_train_steps"]:
            break
        
        batch_end = min(batch_start + CONFIG["train_batch_size"], len(indices))
        batch_indices = indices[batch_start:batch_end]
        batch_df = df_train.iloc[batch_indices]
        
        # Prepare batch
        batch = make_batch(
            batch_df["sequence"].tolist(),
            batch_df["label"].tolist(),
            tokenizer,
            CONFIG["max_seq_length"],
            DEVICE
        )
        
        # Forward pass with bfloat16
        with autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Compute accuracy
        logits = outputs.logits.detach()
        preds = torch.argmax(logits, dim=1)
        accuracy = (preds == batch["labels"]).float().mean().item()
        
        # Track metrics
        train_history["step"].append(step)
        train_history["loss"].append(loss.item())
        train_history["accuracy"].append(accuracy)
        
        pbar.set_postfix({"loss": f"{loss.item():.3f}", "acc": f"{accuracy:.3f}"})
        
        # Validation
        if (step + 1) % CONFIG["eval_every_n_steps"] == 0 and len(df_val) > 0:
            val_loss, val_acc = evaluate(model, df_val, tokenizer, CONFIG, DEVICE)
            val_history["step"].append(step)
            val_history["loss"].append(val_loss)
            val_history["accuracy"].append(val_acc)
            print(f"  [Step {step+1}] Val Loss: {val_loss:.3f}, Val Acc: {val_acc:.3f}")
            model.train()
        
        step += 1

print(f"\n✓ Training completed: {step} steps")

## Metrics & Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot training loss with rolling average
ax = axes[0]
ax.plot(train_history["step"], train_history["loss"], alpha=0.3, linewidth=1, label="Raw")
# Rolling average (20 steps)
if len(train_history["loss"]) >= 20:
    rolling_loss = pd.Series(train_history["loss"]).rolling(window=20).mean()
    ax.plot(train_history["step"], rolling_loss, linewidth=2, label="Rolling avg (20 steps)")
if val_history["loss"]:
    ax.plot(val_history["step"], val_history["loss"], 'o-', linewidth=2, markersize=6, label="Validation")
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot accuracy
ax = axes[1]
ax.plot(train_history["step"], train_history["accuracy"], alpha=0.3, linewidth=1, label="Raw")
if len(train_history["accuracy"]) >= 20:
    rolling_acc = pd.Series(train_history["accuracy"]).rolling(window=20).mean()
    ax.plot(train_history["step"], rolling_acc, linewidth=2, label="Rolling avg (20 steps)")
if val_history["accuracy"]:
    ax.plot(val_history["step"], val_history["accuracy"], 'o-', linewidth=2, markersize=6, label="Validation")
ax.set_xlabel("Training Step")
ax.set_ylabel("Accuracy")
ax.set_title("Training & Validation Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_metrics.png", dpi=150, bbox_inches='tight')
plt.show()

print("✓ Metrics plots saved")

## Save Fine-Tuned Model

In [ ]:
# Save LoRA adapters
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapters")
print(f"✓ LoRA adapters saved to {OUTPUT_DIR}/lora_adapters")

# Save training config and history
with open(f"{OUTPUT_DIR}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

# Save training history
df_train_hist = pd.DataFrame(train_history)
df_train_hist.to_csv(f"{OUTPUT_DIR}/train_history.csv", index=False)

if val_history["loss"]:
    df_val_hist = pd.DataFrame(val_history)
    df_val_hist.to_csv(f"{OUTPUT_DIR}/val_history.csv", index=False)

print(f"\n✓ All outputs saved:")
print(f"  - {OUTPUT_DIR}/lora_adapters/")
print(f"  - {OUTPUT_DIR}/config.json")
print(f"  - {OUTPUT_DIR}/train_history.csv")
print(f"  - {OUTPUT_DIR}/val_history.csv")
print(f"  - {OUTPUT_DIR}/training_metrics.png")

## Inference with Fine-Tuned Model

In [ ]:
# Example: Use fine-tuned model for prediction
model.eval()

test_sequences = [
    "MKVLIVAALLLAVGLAFWECEKRKYQCPEKPQE",
    "MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTG",
]

with torch.no_grad():
    for seq in test_sequences:
        batch = make_batch([seq], [0], tokenizer, CONFIG["max_seq_length"], DEVICE)  # dummy label
        with autocast(dtype=torch.bfloat16):
            outputs = model(**{k: v for k, v in batch.items() if k != "labels"})
        
        logits = outputs.logits[0]
        probs = F.softmax(logits, dim=-1)
        pred_class = torch.argmax(probs).item()
        pred_prob = probs[pred_class].item()
        
        print(f"Sequence: {seq[:30]}...")
        print(f"  Predicted class: {pred_class} ({EC_CLASSES[pred_class]})")
        print(f"  Confidence: {pred_prob:.3f}")
        print()

## Usage Guide

In [ ]:
display(Markdown("""
## How to Use Fine-Tuned Model

### **Custom Training Data**
1. Prepare CSV with columns: `sequence`, `label`
2. Load: `df_data = pd.read_csv("your_data.csv")`
3. Update `CONFIG["num_labels"]` to your number of classes
4. Run training cells

### **Load Fine-Tuned Model Later**

```python
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer

base_model = AutoModelForSequenceClassification.from_pretrained(
    "biohub/ESMC-300M",
    num_labels=7
)
model = PeftModel.from_pretrained(base_model, "/content/esmc_finetuned/lora_adapters")
model.eval()
```

### **Tasks You Can Train**
- **Classification**: Change `num_labels` and load data
- **Regression**: Use MSE loss instead of cross-entropy
- **Multi-label**: Modify loss function
- **Other domains**: Enzyme activity, protein stability, localization, etc.

### **Hyperparameter Tuning**
- **Learning rate**: 1e-4 (default), try 1e-5 or 5e-5
- **LoRA rank**: 8 (default), try 16 for more capacity
- **Batch size**: 8 (default), increase to 16 if GPU allows
- **Number of steps**: Increase from 1000 for better convergence

### **Performance Tips**
- LoRA is memory-efficient: runs on A100/H100 with batch size 8-16
- Use bfloat16 mixed precision (already enabled)
- Validate every 250 steps to catch overfitting early
- Save best checkpoint if validation acc improves
"""))